# Analisis exploratorio - Art Institute of Chicago

Este notebook documenta el flujo de Ciencia de Datos solicitado en el taller: **extract -> raw -> transform -> EDA**.

La ingesta se realiza en `ingesta.py`, donde los documentos JSON crudos se almacenan en MongoDB. En este notebook se leen esos datos RAW, se seleccionan variables relevantes, se aplican transformaciones minimas y se generan insights y graficos.

## Resumen ejecutivo e indice

- **Fuente**: Art Institute of Chicago API.
- **Almacenamiento RAW**: MongoDB, base `taller4_db`, coleccion `raw_data`.
- **Precondicion**: ejecutar `python ingesta.py` antes de correr este notebook.
- **Objetivo del EDA**: analizar obras de arte por artista, fecha, departamento, tipo de obra, origen y medio.
- **Salidas esperadas**: validaciones, DataFrame limpio, minimo 5 insights y 3 graficos.

## 1. Configuracion y conexion a MongoDB

Se cargan las librerias y variables de entorno. Si no existe archivo `.env`, se usan los valores por defecto requeridos por la guia del taller.

In [ ]:
import os

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", 50)

In [ ]:
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
MONGO_DB = os.getenv("MONGO_DB", "taller4_db")
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION", "raw_data")

client = MongoClient(MONGO_URI)
collection = client[MONGO_DB][MONGO_COLLECTION]

print(f"Conexion configurada para MongoDB: {MONGO_DB}.{MONGO_COLLECTION}")

## 2. Validacion post-carga

Esta seccion verifica que la etapa de ingesta haya dejado suficientes documentos en MongoDB. Tambien revisa unicidad por `id`, que funciona como llave natural de la API.

In [ ]:
total_documents = collection.count_documents({})
unique_api_ids = len(collection.distinct("id"))
duplicate_api_ids = total_documents - unique_api_ids
sample_document = collection.find_one({}, {"_id": 0})

validation_summary = pd.DataFrame({
    "metrica": ["documentos_raw", "ids_unicos", "posibles_duplicados"],
    "valor": [total_documents, unique_api_ids, duplicate_api_ids],
})

validation_summary

In [ ]:
if total_documents < 100:
    raise ValueError("La coleccion debe tener minimo 100 documentos. Ejecuta primero python ingesta.py")

if duplicate_api_ids != 0:
    raise ValueError("Se detectaron ids duplicados. Revisa la logica de upsert en ingesta.py")

print("Validacion correcta: hay minimo 100 documentos y no se detectan ids duplicados.")

In [ ]:
sample_fields = sorted(sample_document.keys()) if sample_document else []
print("Campos disponibles en un documento RAW de ejemplo:")
print(sample_fields[:40])

## 3. Carga del dato RAW en Pandas

MongoDB conserva el documento original de la API. Pandas se usa para crear una vista tabular que facilite el analisis, sin modificar la coleccion RAW.

In [ ]:
records = list(collection.find({}, {"_id": 0}))
raw_df = pd.DataFrame(records)

print(f"Filas RAW: {raw_df.shape[0]}")
print(f"Columnas RAW: {raw_df.shape[1]}")
raw_df.head(3)

## 4. Seleccion de variables para el EDA

A partir del documento crudo se seleccionan variables con valor analitico. Esta es la frontera entre el dato RAW no relacional y una tabla curada para exploracion.

In [ ]:
variable_dictionary = pd.DataFrame([
    {"variable": "id", "descripcion": "Identificador unico de la obra en la API."},
    {"variable": "title", "descripcion": "Titulo de la obra."},
    {"variable": "artist_title", "descripcion": "Nombre del artista o autor registrado."},
    {"variable": "date_start", "descripcion": "Anio inicial asociado a la obra."},
    {"variable": "department_title", "descripcion": "Departamento del museo al que pertenece la obra."},
    {"variable": "artwork_type_title", "descripcion": "Tipo o categoria de obra."},
    {"variable": "place_of_origin", "descripcion": "Lugar de origen registrado."},
    {"variable": "medium_display", "descripcion": "Medio o tecnica de la obra."},
])

variable_dictionary

In [ ]:
selected_columns = [
    "id",
    "title",
    "artist_title",
    "date_start",
    "department_title",
    "artwork_type_title",
    "place_of_origin",
    "medium_display",
]

missing_columns = [column for column in selected_columns if column not in raw_df.columns]
if missing_columns:
    raise KeyError(f"Faltan columnas esperadas en el RAW: {missing_columns}")

df = raw_df[selected_columns].copy()
df.head()

## 5. Transformacion minima y limpieza

La limpieza se limita al DataFrame de analisis. Se normalizan textos vacios, se convierten fechas a numerico y se crean dos variables auxiliares para mejorar los insights.

In [ ]:
unknown_values = {
    "title": "Titulo no identificado",
    "artist_title": "Artista no identificado",
    "department_title": "Sin departamento",
    "artwork_type_title": "Sin tipo",
    "place_of_origin": "Origen no identificado",
    "medium_display": "Medio no identificado",
}

for column, replacement in unknown_values.items():
    df[column] = df[column].fillna(replacement).astype(str).str.strip()
    df.loc[df[column].eq(""), column] = replacement

df["date_start"] = pd.to_numeric(df["date_start"], errors="coerce")

def classify_century(year):
    if pd.isna(year):
        return "Fecha no identificada"
    if year <= 0:
        return "Antes de era comun"
    century = int((year - 1) // 100 + 1)
    return f"Siglo {century}"

df["century"] = df["date_start"].apply(classify_century)
df["has_identified_artist"] = df["artist_title"].ne("Artista no identificado")
df = df.drop_duplicates(subset=["id"]).reset_index(drop=True)

df.head()

## 6. Inspeccion basica

Se revisan primeras filas, tipos de datos y nulos. Esta parte responde directamente al requisito de inspeccion del taller.

In [ ]:
df.head(10)

In [ ]:
df.info()

In [ ]:
null_summary = (
    df.isnull()
    .sum()
    .reset_index(name="nulos")
    .rename(columns={"index": "variable"})
)
null_summary["porcentaje"] = (null_summary["nulos"] / len(df) * 100).round(2)
null_summary

In [ ]:
sanity_checks = pd.DataFrame({
    "revision": [
        "filas_en_dataframe",
        "columnas_en_dataframe",
        "ids_duplicados",
        "artistas_identificados",
        "anio_minimo",
        "anio_maximo",
    ],
    "valor": [
        len(df),
        df.shape[1],
        df.duplicated(subset=["id"]).sum(),
        int(df["has_identified_artist"].sum()),
        int(df["date_start"].min()) if df["date_start"].notna().any() else "No disponible",
        int(df["date_start"].max()) if df["date_start"].notna().any() else "No disponible",
    ],
})

sanity_checks

## 7. Insights numericos y conteos

Los siguientes resultados resumen patrones relevantes de la muestra. Se calculan mas de 5 para tener material suficiente al redactar el PDF final.

In [ ]:
total_obras = len(df)
artistas_unicos = df["artist_title"].nunique()
artistas_identificados = int(df["has_identified_artist"].sum())
porcentaje_artistas_identificados = artistas_identificados / total_obras * 100

department_counts = df["department_title"].value_counts()
type_counts = df["artwork_type_title"].value_counts()
origin_counts = df["place_of_origin"].value_counts()
century_counts = df["century"].value_counts()

valid_years = df["date_start"].dropna()
anio_min = int(valid_years.min()) if not valid_years.empty else "No disponible"
anio_max = int(valid_years.max()) if not valid_years.empty else "No disponible"

insights = [
    f"El conjunto analizado contiene {total_obras} obras de arte.",
    f"Hay {artistas_unicos} artistas o autores distintos en los registros seleccionados.",
    f"El {porcentaje_artistas_identificados:.1f}% de las obras tiene artista identificado.",
    f"El departamento con mas obras es '{department_counts.idxmax()}', con {department_counts.max()} registros.",
    f"El tipo de obra mas frecuente es '{type_counts.idxmax()}', con {type_counts.max()} registros.",
    f"El lugar de origen mas frecuente es '{origin_counts.idxmax()}', con {origin_counts.max()} registros.",
    f"El siglo mas frecuente es '{century_counts.idxmax()}', con {century_counts.max()} obras.",
    f"Las fechas de inicio de las obras van desde {anio_min} hasta {anio_max}.",
]

for number, insight in enumerate(insights, start=1):
    print(f"{number}. {insight}")

## 8. Grafico de torta obligatorio

El grafico muestra la proporcion de obras por departamento. Para mantenerlo legible, se agrupan los departamentos menos frecuentes en `Otros`.

In [ ]:
department_counts = df["department_title"].value_counts()
top_departments = department_counts.head(5)
other_departments = department_counts.iloc[5:].sum()

pie_data = top_departments.copy()
if other_departments > 0:
    pie_data.loc["Otros"] = other_departments

plt.figure(figsize=(8, 8))
plt.pie(pie_data, labels=pie_data.index, autopct="%1.1f%%", startangle=90)
plt.title("Proporcion de obras por departamento")
plt.ylabel("")
plt.tight_layout()
plt.show()

## 9. Grafico libre 1: barras por tipo de obra

Este grafico permite identificar concentracion por categorias artisticas.

In [ ]:
type_counts = df["artwork_type_title"].value_counts().head(10)

sns.barplot(
    x=type_counts.values,
    y=type_counts.index,
    hue=type_counts.index,
    palette="viridis",
    legend=False,
)
plt.title("Top 10 tipos de obra")
plt.xlabel("Cantidad de obras")
plt.ylabel("Tipo de obra")
plt.tight_layout()
plt.show()

## 10. Grafico libre 2: distribucion de fechas

El histograma ayuda a observar en que periodos se concentran las obras de la muestra.

In [ ]:
valid_years = df["date_start"].dropna()

sns.histplot(valid_years, bins=20, kde=True, color="#2a9d8f")
plt.title("Distribucion de fechas de inicio de las obras")
plt.xlabel("Anio de inicio")
plt.ylabel("Cantidad de obras")
plt.tight_layout()
plt.show()

## 11. Criterios de calidad del EDA

- **Completitud**: se valida que existan al menos 100 documentos en MongoDB.
- **Exactitud operativa**: se revisa unicidad por `id` para evitar duplicados.
- **Consistencia**: textos vacios y nulos se normalizan en el DataFrame de analisis.
- **Trazabilidad**: la coleccion `raw_data` conserva el JSON original de la API.
- **Documentacion**: cada etapa del notebook explica que se hace y por que se hace.

## 12. Limitaciones y proximos pasos

La muestra depende de la paginacion y del orden de respuesta de la API. Para un analisis mas profundo se podria aumentar `TARGET_RECORDS`, analizar mas campos del JSON original o comparar obras por periodos historicos, artistas y departamentos especificos.